# nb_03a — Gold: dimensions (SCD1 + two flavours of SCD Type 2)

**Module 3 (dimensions).**

| Dimension | Technique | Why |
|---|---|---|
| `dim_date` | generated | date spine |
| `dim_cost_center` | SCD1 (overwrite) | cost centres are stable; carries the RLS region |
| `dim_pay_band` | **SCD2 from a history feed** | grids re-benchmarked yearly; full history available |
| `dim_worker` | **SCD2 via periodic-snapshot MERGE** | staffing actions change worker records |

Every dimension gets an integer **surrogate key** so the fact can resolve the
*version in force on the event date* (nb_03b).

In [ ]:
from pyspark.sql import functions as F, Window as W
from delta.tables import DeltaTable
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

## 1. `dim_date`

In [ ]:
dates = (spark.sql("""
  SELECT explode(sequence(to_date('2021-01-01'), to_date('2025-12-31'),
                          interval 1 day)) AS date""")
  .withColumn("date_key", F.date_format("date","yyyyMMdd").cast("int"))
  .withColumn("year", F.year("date"))
  .withColumn("fiscal_year",
      F.when(F.month("date")>=4, F.year("date")).otherwise(F.year("date")-1))
  .withColumn("quarter", F.concat(F.year("date"), F.lit("-Q"), F.quarter("date")))
  .withColumn("month", F.date_format("date","yyyy-MM"))
  .withColumn("month_name", F.date_format("date","MMMM")))
(dates.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold.dim_date"))
print(f"dim_date: {dates.count():,} rows")

## 2. `dim_cost_center` — SCD Type 1

Overwrite current state; add a surrogate key. Carries `hr_region`, the row-level
security driver used in Module 6.

In [ ]:
cc = spark.table("bronze.cost_centers")
dim_cost_center = (cc.withColumn("cost_center_key", F.row_number().over(W.orderBy("cost_center_id")))
    .select("cost_center_key","cost_center_id","cost_center_name","branch",
            "hr_region","business_line"))
(dim_cost_center.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold.dim_cost_center"))
print("dim_cost_center:", dim_cost_center.count())

## 3. `dim_pay_band` — SCD Type 2 from a history feed

Bronze holds one row per (group, level, effective_date). Derive validity per
(group, level) ordered by date: `effective_to` = day before the next version;
`is_current` on the latest. This is the **history-feed → SCD2** pattern.

In [ ]:
hist = spark.table("bronze.pay_bands")
w = W.partitionBy("classification_group","classification_level").orderBy("band_effective_date")
scd2 = (hist
    .withColumn("effective_from", F.col("band_effective_date"))
    .withColumn("_next", F.lead("band_effective_date").over(w))
    .withColumn("effective_to",
        F.when(F.col("_next").isNull(), F.to_date(F.lit("9999-12-31")))
         .otherwise(F.date_sub("_next",1)))
    .withColumn("is_current", F.col("_next").isNull())
    .drop("_next","band_effective_date","_ingested_at"))
dim_pay_band = (scd2.withColumn("pay_band_key",
        F.row_number().over(W.orderBy("classification_group","classification_level","effective_from")))
    .select("pay_band_key","classification_group","classification_level",
            "band_min","band_mid","band_max","effective_from","effective_to","is_current"))
(dim_pay_band.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold.dim_pay_band"))
print(f"dim_pay_band: {dim_pay_band.count():,} versioned rows")
dim_pay_band.filter("classification_group='PA' AND classification_level=3") \
    .orderBy("effective_from") \
    .select("band_mid","effective_from","effective_to","is_current").show(truncate=False)

## 4. `dim_worker` — SCD Type 2 via periodic-snapshot MERGE

The canonical **close-then-insert** MERGE, tracking `classification_group`,
`classification_level`, `directorate`, `employment_type` via a `row_hash`.

1. **Initial load** — every worker → version 1 (`is_current=true`,
   `effective_to=9999-12-31`).
2. **Second snapshot** (`workers_delta`, effective 2023-07-01) — a MERGE that
   **closes** changed rows and **inserts** the new version; brand-new employees
   are inserted fresh.

In [ ]:
def hash_cols(*cols):
    return F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c),F.lit("")) for c in cols]),256)
TRACK = ["classification_group","classification_level","directorate","employment_type"]

snap1 = spark.table("bronze.workers")
init = (snap1
    .withColumn("classification_level", F.col("classification_level").cast("int"))
    .withColumn("row_hash", hash_cols(*TRACK))
    .withColumn("effective_from", F.to_date("snapshot_date"))
    .withColumn("effective_to", F.to_date(F.lit("9999-12-31")))
    .withColumn("is_current", F.lit(True))
    .select("employee_id","full_name","home_cost_center_id",*TRACK,"row_hash",
            "effective_from","effective_to","is_current"))
(init.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold.dim_worker"))
print(f"dim_worker initial: {init.count():,} rows")

In [ ]:
snap2 = (spark.table("bronze.workers_delta")
    .withColumn("classification_level", F.col("classification_level").cast("int"))
    .withColumn("row_hash", hash_cols(*TRACK))
    .withColumn("effective_from", F.to_date("snapshot_date")))
tgt = DeltaTable.forName(spark, "gold.dim_worker")
# Step 1: CLOSE changed current rows
(tgt.alias("t").merge(snap2.alias("s"),
        "t.employee_id = s.employee_id AND t.is_current = true")
   .whenMatchedUpdate(condition="t.row_hash <> s.row_hash",
        set={"is_current": F.lit(False),
             "effective_to": F.expr("date_sub(s.effective_from, 1)")})
   .execute())
# Step 2: INSERT new versions (changed) + brand-new employees
current = spark.table("gold.dim_worker").filter("is_current = true")
changed_or_new = (snap2.alias("s")
    .join(current.alias("c"), "employee_id", "left")
    .where("c.employee_id IS NULL OR c.row_hash <> s.row_hash")
    .select("s.employee_id","s.full_name","s.home_cost_center_id",
            *[f"s.{x}" for x in TRACK],"s.row_hash","s.effective_from",
            F.to_date(F.lit("9999-12-31")).alias("effective_to"),
            F.lit(True).alias("is_current")))
changed_or_new.write.format("delta").mode("append").saveAsTable("gold.dim_worker")
spark.sql("""SELECT is_current, count(*) rows FROM gold.dim_worker
             GROUP BY is_current ORDER BY is_current""").show()

In [ ]:
dim_w = (spark.table("gold.dim_worker")
    .withColumn("worker_key", F.row_number().over(W.orderBy("employee_id","effective_from"))))
(dim_w.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold.dim_worker"))
spark.sql("""
  SELECT employee_id, classification_group, classification_level, directorate,
         effective_from, effective_to, is_current
  FROM gold.dim_worker
  WHERE employee_id IN (SELECT employee_id FROM gold.dim_worker
                        GROUP BY employee_id HAVING count(*)>1)
  ORDER BY employee_id, effective_from""").show(10, truncate=False)